# Explore H3-Partitioned Spectral Features

Round-trip validation for Phase 2: build NDVI/NBR/NDWI zonal stats over H3 res-8
hexagons from Landsat-9 COGs, write Hive-partitioned GeoParquet, and read it back
with geopandas.

This notebook is self-contained — it creates a tiny synthetic scene so it runs
without downloading full Landsat tiles. For real COGs, use the CLI command shown
in the final cell.

In [ ]:
from pathlib import Path

import geopandas as gpd
import h3
import numpy as np
import rasterio
from pyproj import Transformer
from rasterio.transform import from_origin

from wildfire_geo_ml.features.h3_partition import build_features, read_partitioned_geoparquet
from wildfire_geo_ml.features.indices import LANDSAT9_OFFSET, LANDSAT9_SCALE
from wildfire_geo_ml.ingest.config import load_pipeline_config
from wildfire_geo_ml.ingest.landsat_paths import local_band_path

SCENE_ID = "LC09_L2SP_044032_20240715_20240717_02_T1"
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
CONFIG_PATH = REPO_ROOT / "config" / "pipeline.yaml"
OUTPUT_DIR = REPO_ROOT / "data" / "features" / "notebook_h3_partitioned"
COG_DIR = REPO_ROOT / "data" / "cog_notebook_demo"


def write_scaled_sr_cog(path: Path, reflectance: float, *, width: int = 32, height: int = 32) -> None:
    """Write a small Landsat SR COG in EPSG:32610 for notebook demos."""
    path.parent.mkdir(parents=True, exist_ok=True)
    dn = max(1, round((reflectance - LANDSAT9_OFFSET) / LANDSAT9_SCALE))
    data = np.full((height, width), dn, dtype=np.uint16)
    cell = "882815d0c1fffff"
    lat, lng = h3.cell_to_latlng(cell)
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:32610", always_xy=True)
    center_x, center_y = transformer.transform(lng, lat)
    span_x = width * 30.0
    span_y = height * 30.0
    origin_x = center_x - span_x / 2
    origin_y = center_y + span_y / 2
    transform = from_origin(origin_x, origin_y, 30.0, 30.0)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint16",
        crs="EPSG:32610",
        transform=transform,
    ) as dst:
        dst.write(data, 1)


pipeline = load_pipeline_config(CONFIG_PATH)
print(f"Config loaded: h3_resolution={pipeline.features.h3_resolution}")

In [ ]:
reflectance = {"B3": 0.3, "B4": 0.1, "B5": 0.5, "B7": 0.2}
for band, value in reflectance.items():
    cog_path = local_band_path(COG_DIR, SCENE_ID, band)
    write_scaled_sr_cog(cog_path, value)

study_bbox = (-122.0, 39.5, -121.5, 40.0)
out_path = build_features(
    COG_DIR,
    [SCENE_ID],
    pipeline,
    study_bbox=study_bbox,
    output_dir=OUTPUT_DIR,
)
print(f"Wrote features to {out_path}")

In [ ]:
gdf = read_partitioned_geoparquet(OUTPUT_DIR)
print(f"Rows: {len(gdf)}")
print(f"Columns: {list(gdf.columns)}")
print(gdf[["scene_id", "h3_index", "ndvi_mean", "nbr_mean", "ndwi_mean"]].head())
print(gdf[["ndvi_mean", "nbr_mean", "ndwi_mean"]].describe())

In [ ]:
# Production CLI (requires validated COGs under data/cog/):
# uv run python -m wildfire_geo_ml.features.h3_partition \
#     --config config/pipeline.yaml \
#     --cog-dir data/cog/ \
#     --output data/features/h3_partitioned/